In [ ]:
import sqlite3
conn = sqlite3.connect(r"D:\python project files\bank profitbility\data\sql\banking.db")
with open(r"D:\python project files\bank profitbility\data\sql\schema.sql","r") as f:
    conn.executescript(f.read())
conn.commit()
conn.close()

In [ ]:
import sqlite3
import pandas as pd 
conn = sqlite3.connect(r"D:\python project files\bank profitbility\data\sql\banking.db")
pd.read_csv(r"D:\python project files\bank profitbility\data\Raw\customers.csv").to_sql("customers",conn,if_exists="append",index=False)
pd.read_csv(r"D:\python project files\bank profitbility\data\Raw\accounts.csv").to_sql("accounts",conn,if_exists="append",index=False)
pd.read_csv(r"D:\python project files\bank profitbility\data\Raw\transactions.csv").to_sql("transactions",conn,if_exists="append",index=False)
pd.read_csv(r"D:\python project files\bank profitbility\data\Raw\payments.csv").to_sql("payments",conn,if_exists="append",index=False)

print(pd.read_sql("select count(*) from customers", conn))
print(pd.read_sql("select count(*) from accounts", conn))
print(pd.read_sql("select count(*) from transactions", conn))
print(pd.read_sql("select count(*) from payments",conn))

conn.close()

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    r"D:\python project files\bank profitbility\data\sql\banking.db"
)

df = pd.read_sql("""
WITH credit_revenue AS (
    SELECT 
        a.customer_id,
        SUM(t.amount * 0.02) AS credit_card_revenue
    FROM accounts a
    JOIN transactions t
        ON a.account_id = t.account_id
    WHERE a.account_type = 'credit_card'
    AND t.transaction_type = 'debit'
    GROUP BY a.customer_id
),
loan_revenue AS (
    SELECT
        customer_id,
        SUM(current_balance * (interest_rate / 100.0)) AS loan_revenue
    FROM accounts
    WHERE account_type = 'loan'
    GROUP BY customer_id
)
SELECT
    c.customer_id,
    c.region,
    COALESCE(cr.credit_card_revenue,0) +
    COALESCE(lr.loan_revenue,0) AS total_revenue
FROM customers c
LEFT JOIN credit_revenue cr
    ON c.customer_id = cr.customer_id
LEFT JOIN loan_revenue lr
    ON c.customer_id = lr.customer_id
""", conn)

conn.close()

df.head()

,customer_id,region,total_revenue
0,100001,Alberta,703.159000
1,100002,Ontarion,2331.405906
2,100003,Quebec,4107.421934
3,100004,Quebec,2721.803348
4,100005,Ontarion,5703.281156


In [5]:
import numpy as np 
import sqlite3
risk = pd.read_sql("""
select 
a.customer_id,
avg(p.days_late) as avg_days_late,
sum(p.default_flag) as total_defaults,
count(p.default_flag) as total_payment_records
from payments p 
join accounts a on p.account_id = a.account_id
group by a.customer_id
""",sqlite3.connect(r"D:\python project files\bank profitbility\data\sql\banking.db"))
df = df.merge(risk,on="customer_id",how="left")
df.fillna(0,inplace=True)


In [6]:
df["default_rate"] = np.where(
    df["total_payment_records"] > 0,
    df["total_defaults"]/df["total_payment_records"],0)
df["risk_adjusted_revenue"] = df["total_revenue"] * (1- df["default_rate"])

In [7]:
df['risk_segment'] = np.where(
    (df['avg_days_late'] > 10) | (df['total_defaults'] > 0),
    'High Risk',
    'Low Risk'
)

In [8]:
df['estimated_clv'] = df['total_revenue'] * 3

In [9]:
    df['customer_type'] = np.select([(df['estimated_clv'] >= df['estimated_clv'].median()) & (df['risk_segment'] == 'Low Risk'),
                                     (df['estimated_clv'] >= df['estimated_clv'].median()) & (df['risk_segment'] == 'High Risk')
                                    ],
                                    [ 'High value - safe',
                                      'High value - Risky'
                                    ],
                                    default = 'Low Value'
                                   )

In [10]:
df['early_warning'] = np.where(
    (df['avg_days_late'] > 5) & (df['total_defaults'] == 0),1,0)


In [11]:
# Step 11 - Estimate operational cost
df["servicing_cost"] = 500

In [12]:
#net profit = interest income + credit card fee income - cost of funds - Expected credit loss - servicing cost

df["net_profit"] = df["risk_adjusted_revenue"] - df["servicing_cost"]



In [13]:
df.to_csv(r"D:\python project files\bank profitbility\data\Processsed\revenue_data.csv", index=False)

In [130]:
df.columns

Index(['customer_id', 'region', 'total_revenue', 'current_balance',
       'cost_of_funds', 'avg_days_late', 'total_defaults',
       'total_payment_records', 'default_rate', 'risk_adjusted_revenue',
       'risk_segment', 'estimated_clv', 'customer_type', 'early_warning',
       'servicing_cost', 'net_profit'],
      dtype='object')